# 🎬 OpenPost — Anime Video Generation on Kaggle (starter)

A free-GPU recipe for **AI anime video**, tuned for what Kaggle actually gives you in Oct 2026.

| Resource | What you get |
|---|---|
| Accelerator | **GPU T4 × 2** — 2 × 16 GB, Turing (`sm_75`): fp16 ✅, bf16 ❌, FlashAttention-2 ❌ |
| RAM / vCPU | 29 GB / 4 |
| Session | up to ~9–12 h, **30 GPU-h per week** quota |
| Disk | 20 GB `/kaggle/working` (persisted), scratch in `/kaggle/temp` |

> ⚠️ The free **P100 was retired on 15 Sep 2026**. T4 × 2 is the only free GPU shape left, so everything below is
> written for 16 GB per card, fp32 VAE, and heavy CPU offloading.

**Model: [`Wan-AI/Wan2.2-TI2V-5B`](https://huggingface.co/Wan-AI/Wan2.2-TI2V-5B-Diffusers)** — Alibaba, Apache-2.0.
It is the best open-weights model that actually fits a T4, and Wan 2.2 is the community's clear favourite for
cel-shaded / hand-drawn aesthetics. Single 5B checkpoint does **both** text→video and image→video at 720p/24 fps
(though we run 480p on a T4).

## Before you press Run
1. Right sidebar → **Settings → Accelerator → GPU T4 × 2**
2. Same panel → **Internet → On** (required: weights + `pip install`)
3. *(optional)* **Add-ons → Secrets → Add a secret** → label `HF_TOKEN` → tick it for this notebook
4. **Run all.** First run downloads ~20 GB (5–10 min); after that one 2-second clip is roughly **10–30 min**.


In [ ]:
# ── 1 · Environment check ────────────────────────────────────────────────────
import os, time, json, shutil, subprocess, itertools, pathlib

# Keep ~20 GB of weights OUT of /kaggle/working (that folder is capped at 20 GB and
# every file in it is snapshotted when the session ends). /kaggle/temp is scratch.
os.environ.setdefault("HF_HOME", "/kaggle/temp/hf")
os.makedirs("/kaggle/working/clips", exist_ok=True)


def run(cmd):
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return (out.stdout or out.stderr).strip()


print(run("nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader"))
print("cpu cores:", os.cpu_count())
for p in ("/kaggle/working", "/kaggle/temp"):
    print(f"free on {p}: {shutil.disk_usage(p).free / 2**30:.1f} GiB")

# T4 x2 is two separate cards, not a 32 GB pool — a single pipeline only uses GPU 0.
# See kaggle/README.md for how to run two renders in parallel, one per card.

In [ ]:
# Kaggle ships an old diffusers; Wan 2.2 needs >= 0.35.
%pip install -q -U "diffusers>=0.35.0" "transformers>=4.56.0" "accelerate>=1.6.0" "safetensors>=0.4.5" "imageio[ffmpeg]"
print("installed")

In [ ]:
import torch, diffusers, transformers

print("torch       ", torch.__version__, "| cuda", torch.version.cuda)
print("gpu         ", torch.cuda.get_device_name(0))
print("capability  ", torch.cuda.get_device_capability(0), "→ (7,5) Turing: use fp16, never bf16")
print("vram        ", f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
print("diffusers   ", diffusers.__version__)
print("transformers", transformers.__version__)

# If this cell raised ImportError, do Runtime → Restart Session and re-run from
# cell 3 onward; the pip install above persists for the life of the notebook VM.

## 🔑 Secrets

Kaggle notebooks read secrets from **Add-ons → Secrets** (top menu of the notebook editor), never from this file.
Add them once on your account, then tick them for each notebook. Retrieve with `UserSecretsClient`.

| Label | Needed for | Required? |
|---|---|---|
| `HF_TOKEN` | gated/private HF repos, private LoRAs you host yourself | optional |
| `KAGGLE_API_TOKEN` | only if *GitHub Actions* pushes/runs this kernel (see `kaggle/README.md`) | optional |

Wan 2.2 is Apache-2.0 and ungated, so this notebook runs anonymously. GitHub Actions secrets
(`https://github.com/SriAkhilSJ/OpenPost/settings/secrets/actions`) are a **different** store — they are never visible
to a Kaggle notebook.

In [ ]:
# ── 2 · Secrets + Hugging Face login (optional) ─────────────────────────────
def get_secret(label):
    try:
        from kaggle_secrets import UserSecretsClient   # exists only inside Kaggle
        return UserSecretsClient().get_secret(label)
    except Exception:
        return os.environ.get(label)                   # local / CI fallback

HF_TOKEN = get_secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("Hugging Face: authenticated ✅")
    except Exception as e:
        print("login failed, continuing anonymously:", e)
else:
    print("Hugging Face: anonymous (fine — Wan 2.2 is ungated)")


## 1 · Configure one clip

Everything worth touching lives in this dict. The two constraints that bite people:

* `width` / `height` must be **multiples of 32** (Wan 2.2 VAE ×16, patch ×2)
* `num_frames` must be **4k + 1** (VAE temporal stride 4) → 49, 81, 121 …

Both are snapped automatically below, so wrong-ish numbers are safe.

In [ ]:
# ── 3 · Configuration ───────────────────────────────────────────────────────
CONFIG = dict(
    mode        = "t2v",          # "t2v" = text→video | "i2v" = image→video
    style       = "cel_90s",      # key of STYLE_PRESETS below
    shot        = "a lone swordsman in a dark coat stands on a rain-soaked rooftop at night, neon signs reflected in the puddles",
    camera      = "slow dolly-in, subtle parallax, hair and coat moving in the wind",

    height      = 480,            # 480p is the T4 sweet spot; 704 needs ~3-4x the time
    width       = 832,
    num_frames  = 49,             # ≈2 s @ 24 fps. 81 ≈ 3.4 s
    fps         = 24,

    steps       = 30,             # official default is 50; 25-30 is the T4 sweet spot
    guidance    = 5.0,
    seed        = 1234,

    lora_repo   = None,           # e.g. "yourname/anime-lora-wan22-5b"
    lora_file   = None,           # e.g. "anime_5b.safetensors"
    lora_scale  = 0.9,

    input_image = "/kaggle/working/keyframe.png",   # only read when mode == "i2v"
    out_dir     = "/kaggle/working/clips",
)

MODE       = CONFIG["mode"]
STYLE      = CONFIG["style"]
SHOT       = CONFIG["shot"]
CAMERA     = CONFIG["camera"]
FPS        = CONFIG["fps"]
STEPS      = CONFIG["steps"]
GUIDANCE   = CONFIG["guidance"]
SEED       = CONFIG["seed"]
LORA_REPO  = CONFIG["lora_repo"]
LORA_FILE  = CONFIG["lora_file"]
LORA_SCALE = CONFIG["lora_scale"]
INPUT_IMAGE = CONFIG["input_image"]
OUT_DIR    = CONFIG["out_dir"]


def snap32(x):
    return max(32, int(round(x / 32)) * 32)


def snap4k1(n):
    n = max(5, int(n))
    return n - ((n - 1) % 4)


HEIGHT = snap32(CONFIG["height"])
WIDTH = snap32(CONFIG["width"])
NUM_FRAMES = snap4k1(CONFIG["num_frames"])

pathlib.Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"{WIDTH}x{HEIGHT}, {NUM_FRAMES} frames ≈ {NUM_FRAMES / FPS:.1f}s @ {FPS} fps, {STEPS} steps")


## 2 · Anime prompt recipe

Base checkpoints default to *photorealistic* — the single biggest quality win for anime is a **style prefix** plus a
negative prompt that fights realism. Presets are all-descriptive (no studio or artist names).

In [ ]:
# ── 4 · Style presets + prompt builder ──────────────────────────────────────
STYLE_PRESETS = {
    "cel_90s": "1990s cel animation look, hand-inked clean lineart, flat cel shading, hand-painted background art, subtle film grain",
    "modern_tv": "modern TV anime, crisp digital lineart, two-tone cel shading, saturated palette, rim lighting, detailed painted backgrounds",
    "watercolor_film": "watercolor and gouache anime film look, soft pastel palette, visible paper texture, painterly backgrounds, gentle natural light",
    "shonen_action": "shonen action anime, bold dynamic linework, high-contrast shading, speed lines, impact frames, dramatic low angle",
    "soft_slice": "soft slice-of-life anime, warm afternoon sunlight, clean pastel shading, detailed everyday props, gentle bloom",
}

# Official Wan 2.2 negative prompt + anime-specific anti-realism terms.
BASE_NEGATIVE = (
    "overexposed, static, blurry details, subtitles, image, still, overall grayish, worst quality, low quality, "
    "JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn faces, deformed, "
    "disfigured, distorted limbs, fingers fused together, static image, cluttered background, three legs, "
    "many people in the background, walking backwards"
)
ANIME_NEGATIVE = "photorealistic, 3d render, CGI, live-action footage, watermark, logo, text, signature"


def build_prompt(shot, style="cel_90s", camera=""):
    parts = [STYLE_PRESETS[style], shot.strip(), camera.strip(),
             "highly detailed, cinematic composition, consistent character design"]
    return ", ".join(p for p in parts if p)


def build_negative(extra=ANIME_NEGATIVE):
    return f"{BASE_NEGATIVE}, {extra}"


PROMPT = build_prompt(SHOT, STYLE, CAMERA)
NEGATIVE_PROMPT = build_negative()

print("PROMPT\n  ", PROMPT, "\n")
print("NEGATIVE\n  ", NEGATIVE_PROMPT)


## 3 · Load Wan 2.2 TI2V-5B (fits 16 GB with offloading)

* `fp16` — required on Turing (bf16 silently produces garbage/black frames on T4)
* VAE stays **fp32 + tiled**: fp16 VAE decoding can go black, and tiling keeps the decode peak low
* `enable_model_cpu_offload()` streams the transformer/text-encoder from the 29 GB of RAM

First run downloads ~20 GB into `/kaggle/temp/hf`.

In [ ]:
# ── 5 · Pipeline ────────────────────────────────────────────────────────────
import gc
from diffusers import AutoencoderKLWan, WanPipeline, WanImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

DTYPE = torch.float16
MODEL_ID = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"

# Free anything left over from a previous attempt before loading ~20 GB.
gc.collect()
torch.cuda.empty_cache()

vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
PipeCls = WanImageToVideoPipeline if MODE == "i2v" else WanPipeline
pipe = PipeCls.from_pretrained(MODEL_ID, vae=vae, torch_dtype=DTYPE)

pipe.enable_model_cpu_offload()          # essential: 5B fp16 ≈ 10 GB of weights alone
for fn in ("enable_tiling", "enable_slicing"):
    try:
        getattr(pipe.vae, fn)()
    except Exception:
        pass

if LORA_REPO:
    pipe.load_lora_weights(LORA_REPO, weight_name=LORA_FILE, adapter_name="anime")
    pipe.set_adapters(["anime"], adapter_weights=[LORA_SCALE])
    print(f"LoRA loaded: {LORA_REPO} @ {LORA_SCALE}")
else:
    print("No LoRA — pure base model (already strong at anime with the style prefix)")

print(f"{PipeCls.__name__} ready on {torch.cuda.get_device_name(0)}")


In [ ]:
# ── 6 · render() — one entry point used by every cell below ─────────────────
def render(prompt=None, out_path=None, seed=None, negative=None, image_path=None,
           frames=None, steps=None, height=None, width=None, fps=None):
    prompt    = prompt    or PROMPT
    negative  = negative  or NEGATIVE_PROMPT
    seed      = SEED      if seed is None else seed
    frames    = NUM_FRAMES if frames is None else snap4k1(frames)
    steps     = STEPS     if steps is None else steps
    height    = HEIGHT    if height is None else snap32(height)
    width     = WIDTH     if width is None else snap32(width)
    fps       = FPS       if fps is None else fps
    out_path  = out_path  or f"{OUT_DIR}/{MODE}_{STYLE}_seed{seed}.mp4"

    kwargs = dict(
        prompt=prompt,
        negative_prompt=negative,
        height=height,
        width=width,
        num_frames=frames,
        guidance_scale=GUIDANCE,
        num_inference_steps=steps,
        generator=torch.Generator(device="cpu").manual_seed(seed),
    )
    if MODE == "i2v":
        if not os.path.exists(image_path or INPUT_IMAGE):
            raise FileNotFoundError(
                f"{image_path or INPUT_IMAGE} not found — upload a keyframe or run the SDXL cell in §6"
            )
        kwargs["image"] = load_image(image_path or INPUT_IMAGE)

    t0 = time.time()
    with torch.inference_mode():
        result = pipe(**kwargs)
    mins = (time.time() - t0) / 60

    pathlib.Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    export_to_video(result.frames[0], out_path, fps=fps)
    print(f"✅ {out_path}\n   {mins:.1f} min · {width}x{height} · {frames} frames · {steps} steps · seed {seed}")
    return out_path


## 4 · Render one clip

Rough T4 timings for this config: **~10–30 min** for 49 frames at 832×480 / 30 steps. If you OOM, drop
`width/height` to `640×352` or `num_frames` to 33 before touching anything else.

In [ ]:
# ── 7 · Generate + preview ──────────────────────────────────────────────────
out = render()
from IPython.display import Video, display
display(Video(out, embed=True, html_attributes="controls loop autoplay muted"))


## 5 · Batch render (optional)

Loops prompts × seeds with a wall-clock budget so a long session can't quietly burn your 30 GPU-h/week.
Outputs are zipped so you can pull them from the **Output** tab in one click.

In [ ]:
# ── 8 · Batch ───────────────────────────────────────────────────────────────
PROMPTS = [
    "a mecha pilot climbs into the cockpit as steam vents across the hangar floor",
    "a schoolgirl runs along a seaside road at golden hour, petals drifting past",
    "two rivals face each other on a bridge at dusk, paper lanterns swaying",
]
SEEDS = [1234, 77]
MAX_MINUTES = 90          # hard stop; a T4 session is ~9-12 h but quota is weekly

t0 = time.time()
for i, (shot, seed) in enumerate(itertools.product(PROMPTS, SEEDS), 1):
    if (time.time() - t0) / 60 > MAX_MINUTES:
        print("⏹ wall-clock budget reached — stopping")
        break
    render(build_prompt(shot, STYLE, CAMERA), f"{OUT_DIR}/batch_{i:02d}_seed{seed}.mp4", seed=seed)

shutil.make_archive("/kaggle/working/clips", "zip", OUT_DIR)
print("→ /kaggle/working/clips.zip  (Output tab → download)")


## 6 · Style lock: keyframe → image-to-video (optional, recommended for series work)

Text→video re-invents the character on every clip. For anything serial, generate the **keyframe** as an anime still
(SDXL anime checkpoint), then let Wan 2.2 do image→video — identity and palette stay put.

Requires `mode = "i2v"` for the render. Animagine XL 4.0 is public and ungated; NoobAI-XL / Illustrious are
alternatives if you host them yourself.

In [ ]:
# ── 9 · Anime keyframe with SDXL ────────────────────────────────────────────
from diffusers import StableDiffusionXLPipeline

SDXL_ID = "cagliostrolab/animagine-xl-4.0"
KEYFRAME_TAGS = (
    "1girl, solo, long black hair, dark coat, standing on rooftop, night, rain, city lights, "
    "looking at viewer, upper body, dramatic lighting, "
    "masterpiece, high score, great score, absurdres"
)
KEYFRAME_NEG = "lowres, bad anatomy, bad hands, text, error, missing fingers, extra digit, watermark, signature, photorealistic, 3d"

sd = StableDiffusionXLPipeline.from_pretrained(SDXL_ID, torch_dtype=torch.float16).to("cuda")
sd.enable_vae_slicing()
img = sd(prompt=KEYFRAME_TAGS, negative_prompt=KEYFRAME_NEG,
         width=WIDTH, height=HEIGHT, num_inference_steps=28, guidance_scale=5.0,
         generator=torch.Generator("cpu").manual_seed(SEED)).images[0]
img.save(INPUT_IMAGE)
print("keyframe →", INPUT_IMAGE)

del sd
gc.collect(); torch.cuda.empty_cache()
img


In [ ]:
# ── 10 · Keyframe → video (reloads the pipeline in i2v mode) ────────────────
MODE = CONFIG["mode"] = "i2v"       # render() reads MODE at call time

del pipe
gc.collect(); torch.cuda.empty_cache()

pipe = WanImageToVideoPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=DTYPE)
pipe.enable_model_cpu_offload()
for fn in ("enable_tiling", "enable_slicing"):
    try:
        getattr(pipe.vae, fn)()
    except Exception:
        pass

MOTION = "she turns her head slowly toward the camera, coat and hair moving in the wind, rain falling"
out = render(build_prompt(MOTION, STYLE, "slow push-in"), f"{OUT_DIR}/i2v_keyframe.mp4", image_path=INPUT_IMAGE)
display(Video(out, embed=True, html_attributes="controls loop autoplay muted"))


## 7 · Fast iteration path: AnimateDiff-Lightning (optional)

When you just need to block out motion, 4–8 distilled steps on an SD 1.5 anime checkpoint gives you a 16-frame
512×512 clip in **seconds**, not half an hour. Quality is far below Wan 2.2 — use it for animatics, then re-render
the keepers with §4.

`Lykon/dreamshaper-8` is a placeholder base. For real anime, upload an SD 1.5 checkpoint (ToonYou, MistoonAnime,
AnythingV5 …) as a private Kaggle dataset and point `BASE_CKPT` at the local path — Civitai downloads need a login.

In [ ]:
# ── 11 · AnimateDiff-Lightning ──────────────────────────────────────────────
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

STEP = 8                                  # 1 / 2 / 4 / 8-step distilled motion module
BASE_CKPT = "Lykon/dreamshaper-8"         # swap for a local anime SD1.5 checkpoint

adapter = MotionAdapter().to("cuda", torch.float16)
adapter.load_state_dict(
    load_file(hf_hub_download("ByteDance/AnimateDiff-Lightning",
                              f"animatediff_lightning_{STEP}step_diffusers.safetensors"),
              device="cuda")
)
ad = AnimateDiffPipeline.from_pretrained(BASE_CKPT, motion_adapter=adapter,
                                         torch_dtype=torch.float16).to("cuda")
ad.scheduler = EulerDiscreteScheduler.from_config(
    ad.scheduler.config, timestep_spacing="trailing", beta_schedule="linear"
)
ad.enable_vae_slicing()

t0 = time.time()
frames = ad(
    prompt=build_prompt(SHOT, STYLE, CAMERA),
    negative_prompt="low quality, worst quality, blurry, watermark, text, photorealistic, 3d",
    num_frames=16,
    guidance_scale=1.0,                   # Lightning is distilled → CFG must be 1
    num_inference_steps=STEP,
).frames[0]

quick = f"{OUT_DIR}/animatediff_lightning.mp4"
export_to_video(frames, quick, fps=8)
print(f"✅ {quick} in {time.time() - t0:.1f}s")
display(Video(quick, embed=True, html_attributes="controls loop autoplay muted"))


## 8 · Why this model, and what it costs

| Model | Params | Fits T4 (16 GB)? | Anime quality | Verdict |
|---|---|---|---|---|
| **Wan 2.2 TI2V-5B** | 5B | ✅ with offload | ★★★★☆ | **Use this.** Apache-2.0, T2V + I2V in one checkpoint, 720p24 capable |
| Wan 2.2 I2V/T2V-A14B + anime LoRA | 27B MoE (14B active) | ❌ 24–80 GB | ★★★★★ | Best anime on the board, but 10–40× slower on T4 even in GGUF/block-swap — rent a 24 GB+ card for finals |
| Wan 2.2 Animate-14B | 14B | ❌ | ★★★★★ | Character/motion transfer from a driving video — the "make my OC dance" tool, needs 24 GB+ |
| LTX-2.x | 22B | ❌ (32 GB FP8) | ★★☆☆☆ | Fast + native audio, weak stylisation, custom licence |
| HunyuanVideo 1.5 | 8–13B | ⚠️ 24 GB comfortable | ★★★☆☆ | Cinematic realism, not anime |
| AnimateDiff-Lightning + anime SD1.5 | 1.8B | ✅ easily | ★★★☆☆ | Seconds per clip, 512×512/16 frames — animatics only |

### Honest limits of the free tier
* 30 GPU-h/week ≈ **60–150 finished 2-second clips** at 480p. Budget accordingly.
* T4 has no bf16 and no FlashAttention-2 → expect ~1/3 of the speed of a 4090, and pin fp16 everywhere.
* 20 GB `/kaggle/working` cap: keep weights in `/kaggle/temp`, and only keep mp4/zip in working.
* Two T4s are **not** one 32 GB GPU. For 2× throughput run two processes with
  `CUDA_VISIBLE_DEVICES=0` / `=1` (see `kaggle/README.md`).

### Next steps worth doing
1. Train or rent a **Wan 2.2 LoRA** for your own house style (16 GB is enough for LoRA training, but not on a T4 within quota).
2. Add `Wan2.2-Animate-14B` on a rented 24 GB card for real character motion.
3. Upscale finished 480p clips with a video upscaler (Real-ESRGAN) as a cheap final pass.
